In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

In [2]:
asset_name = ['SPY', 'QQQ', '^SET.BK', 'GC=F', 'DIA', 'CL=F', '^HSI', '^N225', '^FTSE'] 
#'000001.SS'(SSEC(China)), 'XAU'(Gold) removed due to not found exchanged currency, change 'QQQM' into 'QQQ' because too little data, 'ETC-USD'(Etherium) and 'BTC-USD'(bitcoin)  also removed
#S&P 500, Nasdaq, SET index th, Bitcoint, Gold future, Dow Jones,  Crude Oil future, Hang Seng index, Nikkei 225, FTSE(British)
assetN=len(asset_name)

date_start='1900-01-01'
date_end='2026-08-01'

In [3]:
# download trend data of each asset
trend_data=[] #data
asset_cur=[] #currency

for name in asset_name:
    data=yf.Ticker(name)
    asset_cur.append(data.info['currency'])
    df=data.history(start=date_start, end=date_end)
    df.reset_index(inplace=True)
    df['Date']=df['Date'].apply(lambda x:x.date())

    #temp=np.array(df['Close']+df['Dividends'])
    temp=np.array(df['Close'])
    df2=pd.DataFrame({'date':df['Date'], name:temp})
    trend_data.append(df2)

In [4]:
print("\t asset currency")
print(" | ".join(asset_cur))

print("\t data length")
print(" | ".join([str(len(trend_data[i])) for i in range(assetN)]))

	 asset currency
USD | USD | THB | USD | USD | USD | HKD | JPY | GBP
	 data length
8433 | 6891 | 7218 | 6503 | 7177 | 6512 | 9767 | 15137 | 10756


In [5]:
trend_data[np.argmin([len(i) for i in trend_data])]

,date,GC=F
0,2000-08-30,273.899994
1,2000-08-31,278.299988
2,2000-09-01,277.000000
3,2000-09-05,275.799988
4,2000-09-06,274.200012
...,...,...
6498,2026-07-27,4074.500000
6499,2026-07-28,4036.300049
6500,2026-07-29,4034.699951
6501,2026-07-30,4100.100098


In [6]:
# download data of exchange rate
asset_curr2=list(set([i for i in asset_cur if i != "THB"]))

for name in asset_curr2:
    data=yf.Ticker(name+"THB=X")
    df=data.history(start=date_start, end=date_end)
    df.reset_index(inplace=True)
    df['Date']=df['Date'].apply(lambda x:x.date())

    df2=pd.DataFrame({'date':df['Date'], name:df['Close']})
    trend_data.append(df2)

In [7]:
#merge all trend data
trend_all0=trend_data[0]
for i in range(1,len(asset_name)+len(asset_curr2)):
    trend_all0=pd.merge(trend_all0,trend_data[i], how='outer')
trend_all0.dropna(inplace=True)
trend_all0.reset_index(inplace=True, drop=True)
trend_all0

,date,SPY,QQQ,^SET.BK,GC=F,DIA,CL=F,^HSI,^N225,^FTSE,JPY,HKD,USD,GBP
0,2003-12-01,70.905777,30.250608,641.150024,402.700012,60.294922,29.950001,12456.990234,10403.269531,4410.000000,0.365400,5.136600,39.889999,68.510002
1,2003-12-02,70.727837,30.082094,646.640015,403.700012,60.082031,30.780001,12412.230469,10410.150391,4378.899902,0.366870,5.137200,39.848999,68.930000
2,2003-12-03,70.615799,29.702902,659.429993,403.899994,60.203667,31.100000,12361.179688,10326.389648,4392.000000,0.367040,5.120600,39.806999,68.769997
3,2003-12-04,70.905777,30.023096,659.289978,403.299988,60.507874,31.260000,12342.650391,10429.990234,4378.200195,0.368170,5.132300,39.869999,68.519997
4,2003-12-08,70.886002,29.694477,664.359985,406.600006,60.726963,32.099998,12177.440430,10045.339844,4359.799805,0.370550,5.126700,39.811001,69.019997
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4695,2026-07-13,749.169983,711.739990,1627.900024,3997.000000,524.321045,78.139999,24213.720703,67242.726562,10498.299805,0.205807,4.249946,33.330002,44.639332
4696,2026-07-14,751.830017,719.690002,1626.030029,4061.100098,524.541016,79.339996,24340.730469,67743.500000,10529.400391,0.206085,4.270567,33.509998,44.686565
4697,2026-07-15,754.809998,717.739990,1630.209961,4044.000000,525.800720,79.599998,24681.099609,68751.507812,10515.900391,0.206181,4.266649,33.450001,44.798050
4698,2026-07-16,750.719971,705.940002,1635.290039,3985.600098,524.681030,78.949997,25008.599609,66835.539062,10572.200195,0.207018,4.280556,33.532001,45.440056


In [8]:
# convert all currency into THB
trend_all=trend_all0.iloc[:,:1+len(asset_name)]
for i in range(len(asset_name)):
    if asset_cur[i]!="THB":
        trend_all.iloc[:,1+i]=trend_all0.iloc[:,1+i]*trend_all0.loc[:,asset_cur[i]]
trend_all

,date,SPY,QQQ,^SET.BK,GC=F,DIA,CL=F,^HSI,^N225,^FTSE
0,2003-12-01,2828.431400,1206.696752,641.150024,16063.703241,2405.164397,1194.705512,63986.576256,3801.354546,302129.109421
1,2003-12-02,2818.433492,1198.741342,646.640015,16087.041392,2394.208805,1226.552217,63764.108858,3819.171729,301837.571605
2,2003-12-03,2811.003053,1182.383390,659.429993,16078.046737,2396.527312,1237.997691,63296.659471,3790.198142,302037.825256
3,2003-12-04,2827.013252,1197.020809,659.289978,16079.570083,2412.448853,1246.336176,63346.183366,3840.009433,299994.262685
4,2003-12-08,2822.042668,1182.166852,664.359985,16187.153178,2417.601176,1277.933066,62430.082931,3722.300746,300913.367884
...,...,...,...,...,...,...,...,...,...,...
4695,2026-07-13,24969.836902,23722.295178,1627.900024,133220.017319,17475.621387,2604.406323,102907.008290,13839.023852,468637.088502
4696,2026-07-14,25193.822611,24116.810774,1626.030029,136087.457456,17577.368553,2658.683144,103948.718843,13960.918958,470522.739170
4697,2026-07-15,25248.394994,24008.403221,1630.209961,135271.803085,17588.034492,2662.620010,105305.583275,14175.254952,471091.830724
4698,2026-07-16,25173.143180,23671.581218,1635.290039,133645.148435,17593.605092,2647.351416,107050.716160,13836.159817,480401.367302


In [9]:
trend_all.to_excel('1_trend_data.xlsx', index=False)